# SIMSOPT Stage-2 Coils

## What you will learn
Why plasma-boundary optimization is incomplete until coils reproduce the target field with acceptable complexity.

## Codes used
SIMSOPT in research mode; synthetic coil curves and normal-field-error maps by default.

## Run mode
This notebook uses RUN_MODE = "cached" by default. Allowed values are "tiny", "cached", and "research".

## Expected outputs
Initial/final coil plots, `04_bdotn_before_after.png`, and `04_coil_tradeoff.png`.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "sos2026").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Keep RUN_MODE='cached' first; install requirements-colab.txt from the cloned repo if needed.")
else:
    print("Local runtime detected.")

In [ ]:
RUN_MODE = "cached"  # allowed: "tiny", "cached", "research"
print(f"RUN_MODE = {RUN_MODE}")

In [ ]:
import importlib
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sos2026.paths import PROJECT_ROOT, DATA_DIR, CACHE_DIR, FIGURE_DIR, MOVIE_DIR, ensure_directories
ensure_directories()
print("Figures:", FIGURE_DIR)
print("Cached data:", CACHE_DIR)

## 1. Learning frame

This notebook is a deliberately small project: define one metric, produce one plot, expose one failure mode, and identify where a real code would enter.

In [ ]:
from sos2026.coil_helpers import coil_curves, normal_field_error, tradeoff_table
from sos2026.plotting import savefig, caption

## 2. Load or generate the teaching data

Cached mode uses small arrays so the conceptual workflow is always available.

In [ ]:
tradeoff = tradeoff_table()
tradeoff

## 3. Make the primary plot

Every plot has a one-sentence caption because students should know how to read it without guessing.

In [ ]:
for stage, name in [("initial", "04_initial_coils.png"), ("final", "04_final_coils.png")]:
    fig = plt.figure(figsize=(5.8, 4.2))
    ax = fig.add_subplot(111, projection="3d")
    for curve in coil_curves(stage):
        ax.plot(curve[:, 0], curve[:, 1], curve[:, 2], lw=2)
    ax.set_title(f"{stage.capitalize()} stage-2 coils")
    ax.set_axis_off()
    savefig(fig, name)
    plt.show()
print("Caption: these coils are cached geometry for teaching the stage-2 inverse problem.")

## 4. Probe the metric

A metric becomes useful for optimization only when we understand how it changes across design choices.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.7), sharex=True, sharey=True)
for ax, stage in zip(axes, ["initial", "final"]):
    theta, zeta, err = normal_field_error(stage)
    im = ax.contourf(zeta, theta, err, levels=24, cmap="coolwarm")
    ax.set_title(stage)
    ax.set_xlabel("zeta")
    ax.set_ylabel("theta")
fig.colorbar(im, ax=axes.ravel().tolist(), label="B dot n proxy")
fig.savefig(FIGURE_DIR / "04_bdotn_before_after.png", dpi=160, bbox_inches="tight")
plt.show()
print("Caption: stage-2 optimization tries to reduce normal field error without producing unbuildable coils.")

## 5. Interpret the design consequence

The table below translates the plot into an optimization decision.

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.7))
ax.plot(tradeoff["coil_length"], tradeoff["quadratic_flux"], marker="o", lw=2)
for _, row in tradeoff.iterrows():
    ax.text(row["coil_length"] + 0.005, row["quadratic_flux"], f"w={row['weight_length']}", fontsize=8)
ax.set_xlabel("normalized coil length")
ax.set_ylabel("quadratic flux proxy")
ax.set_title("Regularization tradeoff")
ax.grid(alpha=0.25)
caption(ax, "Shorter coils can damage field accuracy, so stage-2 design is multiobjective.")
savefig(fig, "04_coil_tradeoff.png")
plt.show()

## 6. Failure mode

The cached plot is useful only if we say what it does not prove.

In [ ]:
failure_mode = pd.DataFrame({
    "cached_mode_proves": ["workflow shape", "plot grammar", "where the metric enters"],
    "cached_mode_does_not_prove": ["validated physics", "final design ranking", "runtime scalability"],
})
failure_mode

## 7. Research-mode hook

Run this cell only after timing the package on the lecture machine.

In [ ]:
if RUN_MODE == "research":
    import simsopt
    print("SIMSOPT import OK:", getattr(simsopt, "__version__", "unknown"))
    print("Research path: load data/inputs/simsopt/input.LandremanPaul2021_QA")
else:
    print("Cached mode: research package path skipped intentionally.")

## 8. Mini project handoff

Use this notebook during the lecture as the computational project slide points to: change one parameter, regenerate one plot, and explain one design tradeoff.

In [ ]:
project_steps = pd.DataFrame({
    "step": [1, 2, 3, 4],
    "action": ["identify metric", "change one input", "regenerate plot", "state failure mode"],
})
project_steps

## Try this
Change one scalar or one row in the cached data and regenerate the primary plot.

## Expected qualitative answer
The plot should move in a physically interpretable direction, but the cached result remains an educational proxy.

## Research extension
Replace the cached data source with the corresponding real package output after timing and API verification.